# 05 · Two-Stage Retrieval
**Stage 3 of LPS — Topic + Tone Matching**

- **Tier 1** — Semantic topic match: query vs. Cognitive Anchor centroids (cosine similarity)
- **Tier 2** — Meta-contextual tone match: filter by $\kappa$ (asserting, questioning, etc.)

The two-stream output feeds directly into Dual-Stream Contextualization (Notebook 06).


In [ ]:

import sys
sys.path.append("../src")

import numpy as np
import json, yaml, pickle
from embeddings import UserPrioritizedEmbedder
from clustering import ELCFramework
from retrieval import TwoStageRetriever, TONE_LABELS

with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
with open("../data/cohort.json") as f:
    cohort = json.load(f)
with open("../data/embeddings.pkl", "rb") as f:
    all_embeddings = pickle.load(f)

uid      = list(cohort.keys())[0]
data     = all_embeddings[uid]
embedder = UserPrioritizedEmbedder(alpha=cfg["embedding"]["alpha"])
print(f"User: {uid[:12]}... — {len(data['interactions'])} interactions")


In [ ]:

# ── Fit ELC and prepare retriever ─────────────────────────────────────
elc = ELCFramework(
    lambda_decay=cfg["elc"]["lambda_decay"],
    min_cluster_size=cfg["elc"]["min_cluster_size"],
    min_samples=cfg["elc"]["min_samples"],
)
elc.fit(data["embeddings"], data["timestamps"])

# Annotate interactions with inferred kappa (tone)
retriever = TwoStageRetriever(
    encoder=embedder.encoder,
    top_k_exemplars=cfg["retrieval"]["top_k_exemplars"],
)

# Assign kappa to each stored interaction
for inter in data["interactions"]:
    emb = embedder.encode([inter["user"]])[0]
    inter["kappa"] = retriever._infer_query_tone(emb)
    inter["embedding"] = emb

print(f"Tone distribution:")
from collections import Counter
tones = Counter(inter["kappa"] for inter in data["interactions"])
for tone, count in tones.most_common():
    print(f"  {tone:15s}: {count:4d}  ({100*count/len(data['interactions']):.1f}%)")


In [ ]:

# ── Run two-stage retrieval for a test query ──────────────────────────
test_conv  = list(cohort[uid]["test"])[0]
test_query = test_conv["pairs"][0]["user"]
query_emb  = embedder.encode([test_query])[0]

# Tier 1: get anchor-matched exemplars
tier1_exemplars = elc.retrieve_exemplars(
    query_embedding=query_emb,
    interactions=data["interactions"],
    top_k_anchors=cfg["retrieval"]["top_k_anchors"],
    top_k_exemplars=cfg["retrieval"]["top_k_exemplars"] * 3,  # wider pool for Tier 2
)

# Full two-stage retrieval
result = retriever.retrieve(test_query, query_emb, tier1_exemplars)

print(f"Query: {test_query[:100]}...")
print(f"\nInferred tone κ: {result['query_tone']}")
print(f"\nLOGIC STREAM ({len(result['logic_stream'])} exemplars):")
for i, ex in enumerate(result["logic_stream"], 1):
    print(f"  [{i}] [{ex.get('kappa','?')}] {ex['user'][:70]}...")
print(f"\nSTYLE STREAM ({len(result['style_stream'])} exemplars, κ={result['query_tone']}):")
for i, ex in enumerate(result["style_stream"], 1):
    print(f"  [{i}] {ex['user'][:70]}...")


In [ ]:

# ── Format context string for LLM ────────────────────────────────────
context = TwoStageRetriever.format_context(result)
print("=== Formatted context for LLM generator ===")
print(context)
print("\n✓ Notebook 05 complete — retrieval context ready for generation")
